# Try Mirage: real cloak + simulator

Judge-facing, no API server required. Use a fresh Google Colab **T4 GPU** runtime. Set `REPO_URL` in the setup cell to the Mirage Git clone URL, then run cells from top to bottom. The notebook calls the backend's real cloak and simulator services directly; it uses the fixed simulator prompts and settings from the backend PRD. Model downloads and inference may take several minutes.

Only upload a photo you have permission to process. Outputs are saved in a temporary runtime directory; download the cloaked image in the last cell.

In [ ]:
!nvidia-smi

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Set this to the repository's HTTPS or SSH clone URL before running this cell.
REPO_URL = ""
REPO_DIR = Path("/content/mirage")
if not REPO_URL.strip():
    raise ValueError("Set REPO_URL to the Mirage Git clone URL, then rerun this cell.")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

if (REPO_DIR / "backend" / "app" / "main.py").is_file():
    PROJECT_DIR = REPO_DIR / "backend"
elif (REPO_DIR / "app" / "main.py").is_file():
    PROJECT_DIR = REPO_DIR
else:
    raise FileNotFoundError("The clone must contain app/main.py, either at its root or under backend/.")
requirements = PROJECT_DIR / "requirements.txt"
if not requirements.is_file():
    raise FileNotFoundError(f"Backend requirements file not found: {requirements}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)

os.environ["MOCK"] = "0"
os.environ["DEVICE"] = "auto"
os.environ["DISABLE_SIM"] = "0"
sys.path.insert(0, str(PROJECT_DIR))

import torch
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required. In Colab choose Runtime > Change runtime type > T4 GPU.")
from diffusers import AutoencoderKL, StableDiffusionImg2ImgPipeline
print(f"Using GPU: {torch.cuda.get_device_name(0)}")
print(f"Backend checkout: {PROJECT_DIR}")

In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
import tempfile
import uuid

import matplotlib.pyplot as plt
from google.colab import files
from tqdm.auto import tqdm
from PIL import Image

from app.config import settings
from app.schemas import CloakRequest, Strength, SimPreset
from app.services.cloak import process_cloak
from app.services.simulate import process_simulation

if settings.mode != "gpu":
    raise RuntimeError("Backend settings did not detect a GPU; select a Colab T4 runtime and restart.")

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one photo.")
upload_name, upload_bytes = next(iter(uploaded.items()))
file_id = str(uuid.uuid4())
work_dir = Path(tempfile.mkdtemp(prefix="mirage-try-")) / file_id
work_dir.mkdir(parents=True, exist_ok=True)
expires_at = (datetime.now(timezone.utc) + timedelta(seconds=settings.ttl_seconds)).isoformat().replace("+00:00", "Z")
print(f"Uploaded {upload_name}; processing at up to {settings.max_side_px}px on the long side.")

In [ ]:
cloak_progress = tqdm(total=100, desc="Mirage cloak")
last_cloak_progress = 0

def cloak_callback(stage, progress):
    global last_cloak_progress
    cloak_progress.set_description(stage)
    cloak_progress.update(max(0, progress - last_cloak_progress))
    last_cloak_progress = progress

cloak_result = process_cloak(
    image_bytes=upload_bytes,
    request=CloakRequest(
        strength=Strength.standard,
        face_focus=True,
        robust_mode=True,
        embed_watermark=False,
    ),
    file_id=file_id,
    output_dir=work_dir,
    max_side_px=settings.max_side_px,
    callback=cloak_callback,
    expires_at=expires_at,
)
cloak_progress.close()
original_path = work_dir / "original.png"
cloaked_path = work_dir / "cloaked.png"
heatmap_path = work_dir / "heatmap.png"

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for axis, path, title in zip(axes, [original_path, cloaked_path, heatmap_path], ["Original", "Cloaked", "Difference heatmap"]):
    axis.imshow(Image.open(path).convert("RGB"))
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()
plt.show()

print("Strength: standard")
print("Protection score:", cloak_result.metrics.protection_score)
print("Robustness:", cloak_result.metrics.robustness.model_dump())
print("PSNR (dB):", cloak_result.metrics.visual_quality.psnr_db)
print("SSIM:", cloak_result.metrics.visual_quality.ssim)
print("Metadata report:", cloak_result.metadata_report.model_dump())
print("Face detected:", cloak_result.face_found)
print("SHA-256 cloaked:", cloak_result.hashes.sha256_cloaked)

In [ ]:
# Choose one fixed, benign server-side preset; no free-text prompt is accepted.
SIM_PRESET = SimPreset.background_snow
sim_progress = tqdm(total=100, desc="Simulator")
last_sim_progress = 0

def sim_callback(stage, progress):
    global last_sim_progress
    sim_progress.set_description(stage)
    sim_progress.update(max(0, progress - last_sim_progress))
    last_sim_progress = progress

sim_result = process_simulation(
    file_id=file_id,
    preset=SIM_PRESET,
    output_dir=work_dir,
    callback=sim_callback,
)
sim_progress.close()
edited_original_path = work_dir / f"sim_{SIM_PRESET.value}_original.png"
edited_cloaked_path = work_dir / f"sim_{SIM_PRESET.value}_cloaked.png"

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for axis, path, title in zip(axes, [edited_original_path, edited_cloaked_path], ["Simulator on original", "Simulator on cloaked"]):
    axis.imshow(Image.open(path).convert("RGB"))
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()
plt.show()
print(f"Preset: {sim_result.preset.value}; seed: {sim_result.seed}")
print(f"Disruption score (heuristic): {sim_result.disruption_score}/100")

In [ ]:
files.download(str(cloaked_path))